# Flujo de trabajo de regresión: Predicción de la progresión de la diabetes

## ¿Qué es la regresión?

La **regresión** es un tipo de machine learning supervisado que se usa para predecir **valores numéricos continuos**. A diferencia de la clasificación (que predice categorías), la regresión predice cantidades.

**Ejemplos:**
- Predecir precios de casas
- Pronosticar temperatura
- **Nuestro caso**: Predecir la progresión de la diabetes (una medida cuantitativa)

## Lo que aprenderemos

Este notebook demuestra un **flujo de trabajo completo de machine learning**:

1. **Cargar y explorar datos** - Entender con qué trabajamos
2. **Dividir los datos** - Separar training y test
3. **Entrenar modelos** - Ajustar diferentes algoritmos
4. **Evaluar el rendimiento** - Evaluar modelos con métricas
5. **Visualizar resultados** - Entender las predicciones

## El dataset

Usaremos el **dataset de Diabetes** de scikit-learn:
- **442 pacientes** con diabetes
- **10 features**: edad, sexo, IMC, presión arterial y 6 mediciones de suero sanguíneo
- **Target**: Medida cuantitativa de la progresión de la enfermedad un año después de la línea base

## 1. Importar librerías

In [ ]:
# Manipulación de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Establecer semilla aleatoria para reproducibilidad
np.random.seed(42)

# Mejorar la apariencia de los gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Cargar y explorar los datos

### Cargar el dataset

In [ ]:
# Cargar el dataset de diabetes
dataset = load_diabetes()

# Mostrar la descripción del dataset
print(dataset.DESCR)

### Entender la estructura de los datos

In [ ]:
# Crear DataFrame para una exploración más fácil
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['target'] = dataset.target

print(f"Forma del dataset: {df.shape}")
print(f"Número de muestras: {df.shape[0]}")
print(f"Número de features: {df.shape[1] - 1}")
print(f"\nNombres de features: {list(dataset.feature_names)}")
print(f"\nPrimeras filas:")
df.head()

### Estadísticas básicas

In [ ]:
# Mostrar resumen estadístico
df.describe().round(2)

### Verificar el tipo de la variable target

La variable target es **continua** (números reales), confirmando que este es un **problema de regresión**.

In [ ]:
print(f"Tipo de dato de la variable target: {dataset.target.dtype}")
print(f"Rango del target: {dataset.target.min():.1f} a {dataset.target.max():.1f}")
print(f"Media del target: {dataset.target.mean():.1f}")

### Visualizar la distribución del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(dataset.target, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Progresión de la Enfermedad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de la Variable Target')
axes[0].axvline(dataset.target.mean(), color='red', linestyle='--', label=f'Media: {dataset.target.mean():.1f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(dataset.target)
axes[1].set_ylabel('Progresión de la Enfermedad')
axes[1].set_title('Boxplot de la Variable Target')
axes[1].set_xticklabels(['Target'])

plt.tight_layout()
plt.show()

print("El target es continuo -> Este es un problema de REGRESIÓN")

### Correlaciones entre features

Veamos qué features están más correlacionadas con la progresión de la enfermedad:

In [ ]:
# Calcular correlaciones con el target
correlations = df.corr()['target'].drop('target').sort_values(ascending=False)

# Graficar correlaciones
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in correlations]
correlations.plot(kind='barh', color=colors, edgecolor='black')
plt.xlabel('Correlación con la Progresión de la Enfermedad')
plt.title('Correlaciones de Features con el Target')
plt.axvline(0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop 3 features positivamente correlacionadas:")
print(correlations.head(3))
print("\nTop 3 features negativamente correlacionadas:")
print(correlations.tail(3))

## 3. Dividir los datos: Training vs. Test

### ¿Por qué dividir los datos?

Este es el **principio más importante** del machine learning:

- **Training set (80%)**: Usado para enseñar al modelo
- **Test set (20%)**: Usado para evaluar con datos no vistos

**Crítico**: ¡El modelo nunca debe ver los datos de test durante el training!

### Preparar features y target

In [ ]:
# Separar features (X) y target (y)
X, y = load_diabetes(return_X_y=True)

print(f"Forma de las features: {X.shape}")
print(f"Forma del target: {y.shape}")

### Realizar la división

In [ ]:
# Dividir en training y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para test
    random_state=42     # Para reproducibilidad
)

print("Resumen de la división de datos:")
print("=" * 50)
print(f"Total de muestras: {len(X)}")
print(f"Muestras de training: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Muestras de test: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")
print(f"\nForma de las features:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"\nDistribuciones del target (deberían ser similares):")
print(f"  Media de train: {y_train.mean():.1f}")
print(f"  Media de test: {y_test.mean():.1f}")

### Visualizar la división

In [ ]:
# Crear visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Proporciones de la división
sizes = [len(X_train), len(X_test)]
labels = [f'Train\n{len(X_train)} muestras', f'Test\n{len(X_test)} muestras']
colors = ['#66c2a5', '#fc8d62']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
axes[0].set_title('División Train/Test')

# Comparación de distribuciones
axes[1].hist([y_train, y_test], bins=20, label=['Train', 'Test'], alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Progresión de la Enfermedad')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del Target: Train vs. Test')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Las distribuciones son similares -> Buena división!")

## 4. Entrenar los modelos

Entrenaremos dos modelos de regresión diferentes:

1. **Linear Regression**: Asume una relación lineal entre features y target
2. **Decision Tree Regressor**: Puede capturar patrones no lineales

### Modelo 1: Linear Regression

In [ ]:
# Crear y entrenar el modelo de Linear Regression
model_linear = LinearRegression()
model_linear.fit(X_train, y_train)

# Hacer predicciones en el test set
y_pred_linear = model_linear.predict(X_test)

print(f"Forma de los coeficientes del modelo: {model_linear.coef_.shape}")

### Modelo 2: Decision Tree Regressor

In [ ]:
# Crear y entrenar el modelo de Decision Tree
model_tree = DecisionTreeRegressor(random_state=42, max_depth=5)
model_tree.fit(X_train, y_train)

# Hacer predicciones en el test set
y_pred_tree = model_tree.predict(X_test)

print("Decision Tree entrenado correctamente")
print(f"Profundidad del árbol: {model_tree.get_depth()}")
print(f"Número de hojas: {model_tree.get_n_leaves()}")

## 5. Evaluar los modelos

### Métricas de regresión

Usaremos tres métricas comunes:

1. **MAE (Mean Absolute Error)**: Diferencia absoluta promedio entre predicciones y valores reales
   - Cuanto menor, mejor
   - Mismas unidades que la variable target

2. **RMSE (Root Mean Squared Error)**: Raíz cuadrada de las diferencias al cuadrado promedio
   - Cuanto menor, mejor
   - Penaliza más los errores grandes que el MAE

3. **R² Score**: Proporción de varianza explicada por el modelo
   - Rango: 0 a 1 (cuanto mayor, mejor)
   - 1.0 = predicciones perfectas
   - 0.0 = no mejor que predecir la media

In [ ]:
# Calcular métricas para Linear Regression
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))
r2_linear = r2_score(y_test, y_pred_linear)

# Calcular métricas para Decision Tree
mae_tree = mean_absolute_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)

# Mostrar resultados
print("Comparación de rendimiento de modelos")
print("=" * 60)
print(f"{'Métrica':<20} {'Linear Regression':>18} {'Decision Tree':>18}")
print("-" * 60)
print(f"{'MAE':<20} {mae_linear:>18.2f} {mae_tree:>18.2f}")
print(f"{'RMSE':<20} {rmse_linear:>18.2f} {rmse_tree:>18.2f}")
print(f"{'R² Score':<20} {r2_linear:>18.3f} {r2_tree:>18.3f}")
print("=" * 60)

# Determinar el ganador
if r2_linear > r2_tree:
    print("\nLinear Regression tiene mejor rendimiento (R² mayor)")
else:
    print("\nDecision Tree tiene mejor rendimiento (R² mayor)")

### Crear DataFrame de comparación

In [ ]:
# Crear tabla de comparación
results_df = pd.DataFrame({
    'Modelo': ['Linear Regression', 'Decision Tree'],
    'MAE': [mae_linear, mae_tree],
    'RMSE': [rmse_linear, rmse_tree],
    'R² Score': [r2_linear, r2_tree]
})

results_df = results_df.round(3)
results_df

## 6. Visualizar los resultados

### Predicciones vs. Valores Reales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression
axes[0].scatter(y_test, y_pred_linear, alpha=0.6, edgecolors='k')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valores Reales')
axes[0].set_ylabel('Valores Predichos')
axes[0].set_title(f'Linear Regression\nR² = {r2_linear:.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decision Tree
axes[1].scatter(y_test, y_pred_tree, alpha=0.6, edgecolors='k', color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción perfecta')
axes[1].set_xlabel('Valores Reales')
axes[1].set_ylabel('Valores Predichos')
axes[1].set_title(f'Decision Tree\nR² = {r2_tree:.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Gráfico de residuos (para el mejor modelo)

Residuos = Real - Predicho. Idealmente, deberían distribuirse aleatoriamente alrededor de cero.

In [ ]:
# Calcular residuos para Linear Regression (típicamente el mejor modelo)
residuals = y_test - y_pred_linear

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagrama de dispersión de residuos
axes[0].scatter(y_pred_linear, residuals, alpha=0.6, edgecolors='k')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Valores Predichos')
axes[0].set_ylabel('Residuos (Real - Predicho)')
axes[0].set_title('Gráfico de Residuos: Linear Regression')
axes[0].grid(True, alpha=0.3)

# Distribución de residuos
axes[1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residuos')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residuo medio: {residuals.mean():.2f} (debería estar cerca de 0)")
print(f"Desv. estándar de residuos: {residuals.std():.2f}")

### Importancia de features (Decision Tree)

In [ ]:
# Obtener importancias de features del Decision Tree
importances = model_tree.feature_importances_
feature_names = dataset.feature_names

# Crear DataFrame y ordenar
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Graficar
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], edgecolor='black')
plt.xlabel('Importancia')
plt.title('Importancia de Features (Decision Tree)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 3 features más importantes:")
print(importance_df.head(3).to_string(index=False))

## Resumen

### Lo que aprendimos

1. **La regresión predice valores continuos** (vs. la clasificación que predice categorías)
2. **Siempre dividir los datos** en training y test
3. **Diferentes modelos tienen diferentes fortalezas**:
   - Linear Regression: Simple, interpretable, asume linealidad
   - Decision Tree: Puede capturar patrones no lineales, riesgo de overfitting
4. **Usar múltiples métricas** para evaluar (MAE, RMSE, R²)
5. **Las visualizaciones ayudan a entender** el rendimiento del modelo

### Próximos pasos

Compara este flujo de trabajo de **regresión** con el flujo de **clasificación** en el notebook complementario (`diabetes_classification.ipynb`). Nota:
- Estructura similar (cargar, dividir, entrenar, evaluar)
- Diferentes tipos de target (continuo vs. categórico)
- Diferentes métricas (MAE/R² vs. Accuracy/F1)
- Diferentes visualizaciones (diagramas de dispersión vs. confusion matrix)


## 7. Conclusión

En este análisis comparamos un modelo de Linear Regression y un Decision Tree Regressor para predecir la progresión de la diabetes.

### Hallazgos clave:

1.  **Rendimiento de los modelos**:
    - La **Linear Regression** logró un R² de aproximadamente **0.45**, superando significativamente al Decision Tree (R² ≈ 0.33).
    - También tuvo tasas de error más bajas (MAE y RMSE), lo que indica que sus predicciones estaban generalmente más cerca de los valores reales.

2.  **¿Por qué ganó la Linear Regression?**
    - El dataset de Diabetes tiene un número relativamente pequeño de muestras (442). Los modelos más simples suelen rendir mejor en datasets pequeños ya que son menos propensos al overfitting.
    - Las relaciones entre las features fisiológicas (como IMC y presión arterial) y la progresión de la enfermedad son en gran medida lineales, lo que encaja bien con los supuestos de la Linear Regression.

3.  **Análisis de residuos**:
    - Los gráficos de residuos nos ayudan a validar nuestros modelos. Para un buen modelo, los residuos deberían estar dispersos aleatoriamente alrededor de cero.
    - Los patrones en los residuos del Decision Tree (líneas verticales) suelen indicar que el modelo está "agrupando" las predicciones de manera demasiado rígida, una característica común de los modelos basados en árboles con profundidad limitada.

### Próximos pasos para mejorar:
- **Ingeniería de features**: Crear nuevas features (ej., términos de interacción como IMC * Edad) para capturar relaciones más complejas.
- **Ajuste de hiperparámetros**: Usar técnicas como Grid Search para encontrar el `max_depth` óptimo del Decision Tree para prevenir el overfitting o underfitting.
- **Modelos avanzados**: Probar algoritmos más potentes como **Random Forest** o **Gradient Boosting**, que combinan múltiples decision trees para mejorar el rendimiento y la estabilidad.
